# 🥬 LettuceDetect × HaluEval: Hallucination Detection Training & Evaluation

This notebook trains LettuceDetect on the **HaluEval** dataset and produces comprehensive performance analysis.

**Make sure to select GPU runtime**: Runtime → Change runtime type → T4 GPU

### ⚠️ IMPORTANT: Run Cell 1 FIRST, restart runtime, then run all cells from Cell 2

## 1️⃣ Fix Dependencies & Install (Run FIRST, then restart runtime)

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Run this cell FIRST, then restart runtime
#         Runtime → Restart runtime
#         Then SKIP this cell and run from Cell 2 onwards
# ═══════════════════════════════════════════════════════════
!pip install --force-reinstall numpy torch transformers -q
!pip install scikit-learn tqdm matplotlib seaborn -q

# Clone LettuceDetect
import os
if not os.path.exists('LettuceDetect'):
    !git clone https://github.com/KRLabsOrg/LettuceDetect.git

%cd LettuceDetect
!pip install -e . --no-deps -q

print("\n" + "="*50)
print("✅ All packages installed!")
print("")
print("👉 NOW: Go to Runtime → Restart runtime")
print("👉 THEN: Skip this cell, run all cells from Cell 2")
print("="*50)

## 2️⃣ Verify Setup (Run AFTER restart)

In [ ]:
# Navigate to LettuceDetect directory
import os
if os.path.exists('LettuceDetect'):
    os.chdir('LettuceDetect')
print(f"Working dir: {os.getcwd()}")

import torch
import numpy as np
import transformers
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

# Test import
from transformers import AutoTokenizer, AutoModelForTokenClassification
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset
print("\n✅ All imports working!")

## 3️⃣ Download HaluEval Dataset

In [ ]:
os.makedirs("data/halueval", exist_ok=True)

!wget -q -O data/halueval/qa_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/qa_data.json
!wget -q -O data/halueval/dialogue_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/dialogue_data.json
!wget -q -O data/halueval/summarization_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/summarization_data.json

for f in ["qa_data.json", "dialogue_data.json", "summarization_data.json"]:
    size = os.path.getsize(f"data/halueval/{f}") / 1e6
    print(f"  ✅ {f}: {size:.1f} MB")

## 4️⃣ Patch LettuceDetect + Preprocess Data

In [ ]:
# Patch HallucinationSample to accept 'halueval' dataset
import lettucedetect.datasets.hallucination_dataset as hd
from dataclasses import dataclass

@dataclass
class HallucinationSample:
    prompt: str
    answer: str
    labels: list
    split: str
    task_type: str
    dataset: str
    language: str

    def to_json(self):
        return {"prompt": self.prompt, "answer": self.answer, "labels": self.labels,
                "split": self.split, "task_type": self.task_type,
                "dataset": self.dataset, "language": self.language}

    @classmethod
    def from_json(cls, json_dict):
        return cls(prompt=json_dict["prompt"], answer=json_dict["answer"],
                   labels=json_dict["labels"], split=json_dict["split"],
                   task_type=json_dict["task_type"], dataset=json_dict["dataset"],
                   language=json_dict["language"])

hd.HallucinationSample = HallucinationSample
print("✅ Patched HallucinationSample")

In [ ]:
import json, random
from lettucedetect.datasets.hallucination_dataset import HallucinationData

random.seed(42)

def load_jsonl(fp):
    with open(fp, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def make_samples(data, split, prompt_fn, right_key, halluc_key, task):
    samples = []
    for item in data:
        prompt = prompt_fn(item)
        right = item.get(right_key, '')
        if right:
            samples.append(HallucinationSample(prompt, right, [], split, task, 'halueval', 'en'))
        halluc = item.get(halluc_key, '')
        if halluc:
            samples.append(HallucinationSample(prompt, halluc,
                [{'start': 0, 'end': len(halluc), 'label': 'hallucination'}],
                split, task, 'halueval', 'en'))
    return samples

MAX_SAMPLES = 2000
TRAIN_RATIO = 0.8
all_samples = []

configs = [
    ('qa', 'data/halueval/qa_data.json',
     lambda x: f"{x.get('knowledge','')}\n\nQuestion: {x.get('question','')}",
     'right_answer', 'hallucinated_answer'),
    ('dialogue', 'data/halueval/dialogue_data.json',
     lambda x: f"{x.get('knowledge','')}\n\nDialogue History: {x.get('dialogue_history','')}",
     'right_response', 'hallucinated_response'),
    ('summarization', 'data/halueval/summarization_data.json',
     lambda x: x.get('document',''),
     'right_summary', 'hallucinated_summary'),
]

for task, path, prompt_fn, rk, hk in configs:
    data = load_jsonl(path)
    if len(data) > MAX_SAMPLES:
        random.shuffle(data); data = data[:MAX_SAMPLES]
    random.shuffle(data)
    si = int(len(data) * TRAIN_RATIO)
    train = make_samples(data[:si], 'train', prompt_fn, rk, hk, task)
    test = make_samples(data[si:], 'test', prompt_fn, rk, hk, task)
    all_samples.extend(train + test)
    print(f"  {task}: {len(train)} train + {len(test)} test")

hall_data = HallucinationData(samples=all_samples)
with open('data/halueval/halueval_data.json', 'w') as f:
    json.dump(hall_data.to_json(), f, indent=2)

tc = sum(1 for s in all_samples if s.split=='train')
hc = sum(1 for s in all_samples if len(s.labels)>0)
print(f"\n📊 Total: {len(all_samples)} | Train: {tc} | Test: {len(all_samples)-tc}")
print(f"   Hallucinated: {hc} | Supported: {len(all_samples)-hc}")
print("✅ Saved!")

## 5️⃣ Train the Model 🚀

In [ ]:
import json, random, numpy as np, torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset
from lettucedetect.models.trainer import Trainer

random.seed(123); np.random.seed(123); torch.manual_seed(123); torch.cuda.manual_seed_all(123)

hall_data = HallucinationData.from_json(json.loads(Path('data/halueval/halueval_data.json').read_text()))
train_samples = [s for s in hall_data.samples if s.split == 'train']
random.shuffle(train_samples)
dev_size = int(len(train_samples) * 0.1)
dev_samples = train_samples[-dev_size:]
train_samples = train_samples[:-dev_size]
print(f"Train: {len(train_samples)}, Dev: {len(dev_samples)}")

MODEL_NAME = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

train_ds = HallucinationDataset(train_samples, tokenizer)
dev_ds = HallucinationDataset(dev_samples, tokenizer)

BS, EPOCHS, LR = 8, 6, 1e-5
OUTPUT = 'output/halueval_detector'

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, collate_fn=collator)
dev_loader = DataLoader(dev_ds, batch_size=BS, shuffle=False, collate_fn=collator)

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

trainer = Trainer(model=model, tokenizer=tokenizer, train_loader=train_loader,
                  test_loader=dev_loader, epochs=EPOCHS, learning_rate=LR, save_path=OUTPUT)

print(f"\n🚀 Training on {next(model.parameters()).device} | {len(train_loader)} steps/epoch × {EPOCHS} epochs")
trainer.train()
print("\n✅ Training complete!")

## 6️⃣ Evaluation 📊

In [ ]:
import torch, json, numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support, roc_curve, auc, confusion_matrix
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForTokenClassification.from_pretrained('output/halueval_detector', trust_remote_code=True).to(device)
tokenizer = AutoTokenizer.from_pretrained('output/halueval_detector')
collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

hall_data = HallucinationData.from_json(json.loads(Path('data/halueval/halueval_data.json').read_text()))
test_samples = [s for s in hall_data.samples if s.split == 'test']
print(f"📦 Test: {len(test_samples)} | Hall: {sum(1 for s in test_samples if s.labels)} | Supp: {sum(1 for s in test_samples if not s.labels)}")

test_loader = DataLoader(HallucinationDataset(test_samples, tokenizer), batch_size=8, shuffle=False, collate_fn=collator)

model.eval()
tok_p, tok_l, tok_pr = [], [], []
ex_p, ex_l, ex_pr = [], [], []

with torch.no_grad():
    for batch in test_loader:
        out = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        probs = torch.softmax(out.logits, dim=-1)
        preds = torch.argmax(out.logits, dim=-1)
        mask = batch['labels'] != -100
        tok_p.extend(preds[mask].cpu().numpy().tolist())
        tok_l.extend(batch['labels'][mask].cpu().numpy().tolist())
        tok_pr.extend(probs[:,:,1][mask].cpu().numpy().tolist())
        for i in range(batch['labels'].size(0)):
            sl=batch['labels'][i]; sp=preds[i].cpu(); vm=sl!=-100
            if vm.sum()==0: ex_l.append(0);ex_p.append(0);ex_pr.append(0.0)
            else:
                sv=sl[vm].cpu();pv=sp[vm];ppv=probs[i][vm]
                ex_l.append(1 if (sv==1).any().item() else 0)
                ex_p.append(1 if (pv==1).any().item() else 0)
                ex_pr.append(ppv[:,1].max().item())
print("✅ Evaluation done!")

In [ ]:
# ━━━ TOKEN-LEVEL ━━━
pt,rt,f1t,_ = precision_recall_fscore_support(tok_l,tok_p,labels=[0,1],average=None,zero_division=0)
acc_t = accuracy_score(tok_l,tok_p)
fpr_t,tpr_t,_ = roc_curve(tok_l,tok_pr); auroc_t = auc(fpr_t,tpr_t)
cm_t = confusion_matrix(tok_l,tok_p,labels=[0,1])
print("="*60+"\n📊 TOKEN-LEVEL\n"+"="*60)
print(classification_report(tok_l,tok_p,target_names=['Supported','Hallucinated'],digits=4,zero_division=0))
print(f"Accuracy: {acc_t:.4f}  |  AUROC: {auroc_t:.4f}")
print(f"Confusion: TN={cm_t[0][0]} FP={cm_t[0][1]} FN={cm_t[1][0]} TP={cm_t[1][1]}")

# ━━━ EXAMPLE-LEVEL ━━━
pe,re,f1e,_ = precision_recall_fscore_support(ex_l,ex_p,labels=[0,1],average=None,zero_division=0)
acc_e = accuracy_score(ex_l,ex_p)
fpr_e,tpr_e,_ = roc_curve(ex_l,ex_pr); auroc_e = auc(fpr_e,tpr_e)
cm_e = confusion_matrix(ex_l,ex_p,labels=[0,1])
print("\n"+"="*60+"\n📊 EXAMPLE-LEVEL\n"+"="*60)
print(classification_report(ex_l,ex_p,target_names=['Supported','Hallucinated'],digits=4,zero_division=0))
print(f"Accuracy: {acc_e:.4f}  |  AUROC: {auroc_e:.4f}")
print(f"Confusion: TN={cm_e[0][0]} FP={cm_e[0][1]} FN={cm_e[1][0]} TP={cm_e[1][1]}")

In [ ]:
# ━━━ PER-TASK BREAKDOWN ━━━
task_map = {}
for s in test_samples: task_map.setdefault(s.task_type, []).append(s)

for tt, samps in task_map.items():
    tl = DataLoader(HallucinationDataset(samps,tokenizer),batch_size=8,shuffle=False,collate_fn=collator)
    tp,tl2 = [],[]
    with torch.no_grad():
        for b in tl:
            p=torch.argmax(model(b['input_ids'].to(device),attention_mask=b['attention_mask'].to(device)).logits,dim=-1)
            for i in range(b['labels'].size(0)):
                sl=b['labels'][i];sp=p[i].cpu();vm=sl!=-100
                if vm.sum()==0: tl2.append(0);tp.append(0)
                else: tl2.append(1 if (sl[vm].cpu()==1).any().item() else 0);tp.append(1 if (sp[vm]==1).any().item() else 0)
    print(f"\n{'='*50}\n📋 {tt.upper()} ({len(samps)} samples)\n{'='*50}")
    print(classification_report(tl2,tp,target_names=['Supported','Hallucinated'],digits=4,zero_division=0))
    print(f"Accuracy: {accuracy_score(tl2,tp):.4f}")

## 7️⃣ Visualization 📈

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
os.makedirs('output', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax=axes[0][0]
ax.plot(fpr_t,tpr_t,color='#e74c3c',lw=2.5,label=f'AUC={auroc_t:.4f}')
ax.plot([0,1],[0,1],'--',color='#bdc3c7',lw=1.5)
ax.fill_between(fpr_t,tpr_t,alpha=0.1,color='#e74c3c')
ax.set(xlabel='FPR',ylabel='TPR',title='Token-Level ROC');ax.legend();ax.grid(alpha=0.3)

ax=axes[0][1]
ax.plot(fpr_e,tpr_e,color='#3498db',lw=2.5,label=f'AUC={auroc_e:.4f}')
ax.plot([0,1],[0,1],'--',color='#bdc3c7',lw=1.5)
ax.fill_between(fpr_e,tpr_e,alpha=0.1,color='#3498db')
ax.set(xlabel='FPR',ylabel='TPR',title='Example-Level ROC');ax.legend();ax.grid(alpha=0.3)

ax=axes[1][0]
m=['Precision','Recall','F1']; x=np.arange(3); w=0.3
b1=ax.bar(x-w/2,[pt[1],rt[1],f1t[1]],w,label='Token',color='#e74c3c',alpha=0.85)
b2=ax.bar(x+w/2,[pe[1],re[1],f1e[1]],w,label='Example',color='#3498db',alpha=0.85)
ax.set(ylabel='Score',title='Hallucination Detection Metrics',ylim=(0,1.15));ax.set_xticks(x);ax.set_xticklabels(m)
ax.legend();ax.grid(axis='y',alpha=0.3)
for b in list(b1)+list(b2): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{b.get_height():.3f}',ha='center',fontsize=9)

ax=axes[1][1]
sns.heatmap(cm_e,annot=True,fmt='d',cmap='Blues',xticklabels=['Supported','Hallucinated'],
            yticklabels=['Supported','Hallucinated'],ax=ax,annot_kws={'size':14})
ax.set(xlabel='Predicted',ylabel='Actual',title='Confusion Matrix')

plt.suptitle('LettuceDetect on HaluEval — Performance',fontsize=16,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig('output/performance_analysis.png',dpi=150,bbox_inches='tight')
plt.show()
print("✅ Saved!")

In [ ]:
# ━━━ FINAL SUMMARY ━━━
print("\n"+"="*65)
print("   🥬 LETTUCEDETECT × HALUEVAL — PERFORMANCE SUMMARY")
print("="*65)
print(f"\n{'Metric':<28} {'Token-Level':>12} {'Example-Level':>14}")
print("-"*58)
print(f"{'Hallucinated F1':<28} {f1t[1]:>12.4f} {f1e[1]:>14.4f}")
print(f"{'Hallucinated Precision':<28} {pt[1]:>12.4f} {pe[1]:>14.4f}")
print(f"{'Hallucinated Recall':<28} {rt[1]:>12.4f} {re[1]:>14.4f}")
print(f"{'Supported F1':<28} {f1t[0]:>12.4f} {f1e[0]:>14.4f}")
print(f"{'Accuracy':<28} {acc_t:>12.4f} {acc_e:>14.4f}")
print(f"{'AUROC':<28} {auroc_t:>12.4f} {auroc_e:>14.4f}")
print("="*58)

results = {
    'token_level': {'supported':{'p':float(pt[0]),'r':float(rt[0]),'f1':float(f1t[0])},
                    'hallucinated':{'p':float(pt[1]),'r':float(rt[1]),'f1':float(f1t[1])},
                    'accuracy':float(acc_t),'auroc':float(auroc_t)},
    'example_level': {'supported':{'p':float(pe[0]),'r':float(re[0]),'f1':float(f1e[0])},
                      'hallucinated':{'p':float(pe[1]),'r':float(re[1]),'f1':float(f1e[1])},
                      'accuracy':float(acc_e),'auroc':float(auroc_e)},
}
with open('output/evaluation_results.json','w') as f: json.dump(results,f,indent=2)
print("\n✅ Results saved to output/evaluation_results.json")

## 8️⃣ Download Results

In [ ]:
!zip -r halueval_results.zip output/halueval_detector output/evaluation_results.json output/performance_analysis.png
from google.colab import files
files.download('halueval_results.zip')
print("📥 Downloaded!")